## HANDLING LARGE DOCS

### PDFs

**SPLITTING PDF**

In [3]:
import os, requests, io
from urllib.parse import urlparse
from PyPDF2 import PdfReader, PdfWriter 
from pathlib import Path

ORIGINAL DOC_SIZE (BOOK) = 352 PAGES


WE SPLIT IT INTO CHUNKS OF 50 PAGES EACH

In [2]:
#GET PDF_READER GIVEN AN URL (OF A PDF)
def get_pdf_reader(url):
    base_filename = "output"
    response = requests.get(url, stream=True, timeout=30)
    response.raise_for_status()

    #Get filename from URL path
    parsed_url = urlparse(url)
    path_part = os.path.basename(parsed_url.path)
    if path_part and '.' in path_part:
        base_filename = os.path.splitext(path_part)[0]
    
    #Read content into memory
    pdf_content = io.BytesIO(response.content)
    reader = PdfReader(pdf_content)
    total_pages = len(reader.pages)
    return reader, base_filename, total_pages

SPLIT PDF INTO CHUNKS

In [6]:
def split_pdf(url, output_dir, pages_per_chunk):
    reader, base_filename, total_pages = get_pdf_reader(url)

    if reader is None:
        print("Failed to get PDF reader. Aborting split...")
        return

    try:
        #Create the output directory it it doesn't exist
        output_dir = Path(os.path.join(Path.cwd(), output_dir))
        output_dir.mkdir(exist_ok=True)

        #Calculate number of chunks
        num_chunks = (total_pages + pages_per_chunk - 1) // pages_per_chunk
        print(f"Splitting into {num_chunks} chunks of maximum {pages_per_chunk} pages each.")

        #Process chunks
        for i in range(num_chunks):
            writer = PdfWriter()
            start_page = i * pages_per_chunk
            #end_page must be inside total_pages
            end_page = min(start_page + pages_per_chunk, total_pages)

            print(f"Processing chunk {i+1}/{num_chunks} (pages {start_page-1}-{end_page})...")
            
            #Add pages to the new PDF chunk
            for page_num in range(start_page, end_page):
                writer.add_page(reader.pages[page_num])

            #Construct the output filname
            output_filename = os.path.join(output_dir, f"{base_filename}_chunk_{i+1}.pdf")

            #Write chunk to new PDF file
            with open(output_filename, 'wb') as f:
                writer.write(f)
            print(f"Chunk {i+1} saved as '{output_filename}'")

        print("\nPDF splitting completed successfully!")

    except Exception as e:
        print(f"An error occurred during PDF splitting: {str(e)}")



In [7]:
split_pdf("https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf", 
          output_dir='pdf_docs', 
          pages_per_chunk=50)

Splitting into 8 chunks of maximum 50 pages each.
Processing chunk 1/8 (pages -1-50)...
Chunk 1 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_1.pdf'
Processing chunk 2/8 (pages 49-100)...
Chunk 2 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_2.pdf'
Processing chunk 3/8 (pages 99-150)...
Chunk 3 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_3.pdf'
Processing chunk 4/8 (pages 149-200)...
Chunk 4 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_4.pdf'
Processing chunk 5/8 (pages 199-250)...
Chunk 5 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_5.pdf'
Processing chunk 6/8 (pages 249-300)...
Chunk 6 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_6

**HANDLING TABLES & IMAGES**

In [8]:
url = "https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf"
local_file = "sutter_barto.pdf"
with requests.get(url, stream=True, timeout=30) as res:
    res.raise_for_status()
    with open(local_file, 'wb') as f:
        for chunk in res.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

We can analyze this doc using docling

In [13]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

In [21]:
pipeline_options = PdfPipelineOptions(generate_picture_images=True)
res = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
).convert("pdf_docs/SuttonBartoIPRLBook2ndEd_chunk_6.pdf")  # use a 50-page chunk instead of 352 pages

doc = res.document


[INFO] 2026-03-06 15:49:28,824 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-06 15:49:28,828 [RapidOCR] download_file.py:60: File exists and is valid: D:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-06 15:49:28,829 [RapidOCR] main.py:53: Using D:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-06 15:49:28,910 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-06 15:49:28,911 [RapidOCR] download_file.py:60: File exists and is valid: D:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-06 15:49:28,912 [RapidOCR] main.py:53: Using D:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-06 15:49:28,950 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-0

In [23]:
table = doc.tables[0]
table_df = table.export_to_dataframe()
table_df

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


,Program,Hidden Units,Training Games,Opponents,Results
0,TD-Gam 0.0,40,"300,000",other programs,tied for best
1,TD-Gam 1.0,80,"300,000","Robertie, Magriel, ...",- 13 pts / 51 games
2,TD-Gam 2.0,40,"800,000",various Grandmasters,- 7 pts / 38 games
3,TD-Gam 2.1,80,"1,500,000",Robertie,- 1 pt / 40 games
4,TD-Gam 3.0,80,"1,500,000",Kazaros,+6 pts / 20 games


In [25]:
len(doc.tables)

1

# RERANKING WITH CROSSENCODER

In [27]:
from sentence_transformers.cross_encoder import CrossEncoder

In [29]:
query = "What is the main benefit of using a transformer model in NLP?"
documents = [
    "Recurrent Neural Networks (RNNs) were previously popular for sequence tasks.",
    "Transformers allow parallel processing of sequences, unlike RNNs which process tokens sequentially.",
    "The self-attention mechanism in transformers captures long-range dependencies between words effectively.",
    "BERT and GPT are transformer-based models that achieved state-of-the-art results across many NLP benchmarks.",
    "Transformers scale well with data and compute, enabling the training of very large language models.",
    "The main advantage of transformers over LSTMs is their ability to attend to all tokens simultaneously.",
]


In [30]:
model = CrossEncoder('BAAI/bge-reranker-v2-m3')
sentence_pairs = [[query, doc] for doc in documents]
scores = model.predict(sentence_pairs, show_progress_bar=True)

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.58it/s]


In [31]:
docs_with_scores = zip(documents, scores)
reranked = sorted(docs_with_scores, key=lambda x: x[1], reverse=True)
print("\n--- Reranked Documents ---")
print("(Higher score indicates higher relevance)")
for i, (doc, score) in enumerate(reranked):
    print(f"{i+1}. Score {score:.4f} - {doc}")


--- Reranked Documents ---
(Higher score indicates higher relevance)
1. Score 0.8540 - The main advantage of transformers over LSTMs is their ability to attend to all tokens simultaneously.
2. Score 0.4974 - BERT and GPT are transformer-based models that achieved state-of-the-art results across many NLP benchmarks.
3. Score 0.0988 - Transformers scale well with data and compute, enabling the training of very large language models.
4. Score 0.0460 - Transformers allow parallel processing of sequences, unlike RNNs which process tokens sequentially.
5. Score 0.0167 - The self-attention mechanism in transformers captures long-range dependencies between words effectively.
6. Score 0.0000 - Recurrent Neural Networks (RNNs) were previously popular for sequence tasks.
